In [1]:
#IMPLEMENTATION OF 4-2 ENCODER USING REAL DEVICES 


from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import Aer
from qiskit import transpile

def reversible_priority_encoder_minimal(inputs):
    """
    Qubit mapping:
      q[0] = D0 (lowest priority)
      q[1] = D1
      q[2] = D2
      q[3] = D3 (highest priority)
      q[4] = A   (output: D3 OR D2)
      q[5] = B   (output: D3 OR (D1 AND NOT D2))
      q[6] = V   (output: D0 OR D1 OR D2 OR D3)
      q[7] = t1  (ancilla, reused & uncomputed)
      q[8] = t2  (ancilla, reused & uncomputed)
    Classical bits:
      c[0] = A, c[1] = B, c[2] = V
    """
    q = QuantumRegister(9, 'q')
    c = ClassicalRegister(3, 'c')
    qc = QuantumCircuit(q, c)

    # -----------------------
    # Optional: set input bits (example_inputs is a 4-bit tuple/list [D0,D1,D2,D3])
    # -----------------------
    # Set input bits
    for i in range(4):
        if inputs[i] == 1:
            qc.x(q[i])
    # =================================================================
    # Compute A = D3 OR D2  (target q4)
    # reversible OR: cx(a,t); cx(b,t); ccx(a,b,t)
    # =================================================================
    qc.cx(q[3], q[4])      # A ^= D3
    qc.cx(q[2], q[4])      # A ^= D2  (now A = D3 ⊕ D2)
    qc.ccx(q[3], q[2], q[4])# A ^= (D3 & D2) => A = D3 OR D2

    # =================================================================
    # Compute B:
    #   temp t1 = D1 AND NOT D2  (use q7)
    #   B = D3 OR t1  -> store in q5
    # We'll uncompute t1 after using it.
    # =================================================================
    qc.x(q[2])             # flip D2 -> now q2 = NOT D2
    qc.ccx(q[1], q[2], q[7])# t1 = D1 & NOT D2  (q7)
    qc.x(q[2])             # restore D2

    # B = D3 OR t1 into q5
    qc.cx(q[3], q[5])      # B ^= D3
    qc.cx(q[7], q[5])      # B ^= t1
    qc.ccx(q[3], q[7], q[5])# B ^= (D3 & t1) -> q5 = D3 OR t1

    # Uncompute t1 (bring q7 back to 0) - safe because q5 already holds the OR result
    qc.x(q[2])
    qc.ccx(q[1], q[2], q[7])
    qc.x(q[2])
    # now q7 == 0

    # =================================================================
    # Compute V = D0 OR D1 OR D2 OR D3  (target q6)
    # We'll do OR-tree using t1 (q7) and t2 (q8) as temps and then uncompute them.
    # Steps:
    #   t1 = D0 OR D1
    #   t2 = t1 OR D2
    #   V  = t2 OR D3  -> q6
    # Then uncompute t2 and t1.
    # =================================================================
    # t1 = D0 OR D1  -> q7
    qc.cx(q[0], q[7])
    qc.cx(q[1], q[7])
    qc.ccx(q[0], q[1], q[7])

    # t2 = t1 OR D2 -> q8
    qc.cx(q[7], q[8])
    qc.cx(q[2], q[8])
    qc.ccx(q[7], q[2], q[8])

    # V = t2 OR D3 -> q6
    qc.cx(q[8], q[6])
    qc.cx(q[3], q[6])
    qc.ccx(q[8], q[3], q[6])

    # Uncompute t2 (reverse t2 ops)
    qc.ccx(q[7], q[2], q[8])
    qc.cx(q[2], q[8])
    qc.cx(q[7], q[8])
    # Now q8 == 0

    # Uncompute t1 (reverse t1 ops)
    qc.ccx(q[0], q[1], q[7])
    qc.cx(q[1], q[7])
    qc.cx(q[0], q[7])
    # Now q7 == 0

    # -----------------------
    # Measure outputs only (A, B, V)
    # Classical bit order in measurement mapping below: c[0]=A, c[1]=B, c[2]=V
    # Qiskit will print bitstrings in "c2 c1 c0" order, so I'll print mapping clearly below.
    # -----------------------
    qc.measure(q[4], c[0])  # A
    qc.measure(q[5], c[1])  # B
    qc.measure(q[6], c[2])  # V

    return qc
user_input = input("Enter D0 D1 D2 D3 (space separated, 0 or 1): ")
try:
    bits = list(map(int, user_input.split()))
    
    if len(bits) != 4 or any(b not in [0,1] for b in bits):
        raise ValueError
except:
    print("Invalid input! Please enter exactly four 0/1 values.")
    exit()
# print("\nUser Input:")
# for i in range(4):
#     print(f"D{i} = {bits[i]}")

qc = reversible_priority_encoder_minimal(bits)
# #print(qc.data)

# backend = Aer.get_backend('qasm_simulator')
# compiled = transpile(qc, backend)
# result = backend.run(compiled, shots=1).result()
# counts = result.get_counts()

# output = list(counts.keys())[0]


# print("\nRaw bitstring (c2 c1 c0):", output)
# print("Interpreting -> V (c2) B (c1) A (c0)")
# print("A =", output[2], " B =", output[1], " V =", output[0])

# display(qc.draw('mpl'))



In [2]:
#implementation of 4-2 encoder using real devices 



from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit import transpile

# Initialize service (make sure you're logged in first)
service = QiskitRuntimeService()

# Choose least busy backend with >=9 qubits
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=9)
print(f"Running on: {backend.name}")

# Transpile for hardware
transpiled_qc = transpile(qc, backend, optimization_level=1)

options = {"default_shots": 1024}

sampler = Sampler(mode=backend, options=options)
job = sampler.run([transpiled_qc])

print(f"Job submitted! Job ID: {job.job_id()}")

result = job.result()

# Correct way to get counts
counts = result[0].data.c.get_counts()

print("Final Counts:", counts)


qiskit_runtime_service.__init__:WARNING:2026-02-16 23:42:22,204: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-02-16 23:42:23,624: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-02-16 23:42:26,807: Using instance: open-instance, plan: open


Running on: ibm_marrakesh
Job submitted! Job ID: d69ls3d7v0is73fnm0vg
Final Counts: {'110': 101, '111': 718, '011': 77, '000': 14, '101': 66, '010': 14, '100': 6, '001': 28}


In [3]:
# 8->3 reversible priority encoder composed from two reversible 4->2 modules using real devices 
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import Aer
%matplotlib inline


def reversible_4to2(inputs, outA, outB, outV, t1, t2, qc):
    """
    Reversible 4->2 encoder (in-place into outA,outB,outV).
    inputs: tuple/list of 4 qubits [D0,D1,D2,D3] (D3 is highest priority in that group)
    outA, outB, outV: target qubits (must be initially |0>)
    t1, t2: ancilla qubits (must be initially |0>). They will be returned to |0>.
    qc: QuantumCircuit
    """
    d0, d1, d2, d3 = inputs

    # A = D3 OR D2 -> outA
    qc.cx(d3, outA)
    qc.cx(d2, outA)
    qc.ccx(d3, d2, outA)

    # B = D3 OR (D1 AND NOT D2) -> outB
    # compute D1 AND NOT D2 into t1 (use inversion method, then restore D2)
    qc.x(d2)               # flip D2 -> NOT D2 on d2 temporarily
    qc.ccx(d1, d2, t1)     # t1 = D1 & NOT D2
    qc.x(d2)               # restore d2

    # OR with D3 into outB
    qc.cx(d3, outB)
    qc.cx(t1, outB)
    qc.ccx(d3, t1, outB)

    # uncompute t1 (we used d2 inverted earlier; restore t1->0)
    qc.x(d2)
    qc.ccx(d1, d2, t1)
    qc.x(d2)
    # now t1 == 0

    # V = D0 OR D1 OR D2 OR D3 -> outV using t1,t2 as temps
    # t1 = D0 OR D1
    qc.cx(d0, t1)
    qc.cx(d1, t1)
    qc.ccx(d0, d1, t1)

    # t2 = t1 OR D2
    qc.cx(t1, t2)
    qc.cx(d2, t2)
    qc.ccx(t1, d2, t2)

    # outV = t2 OR D3
    qc.cx(t2, outV)
    qc.cx(d3, outV)
    qc.ccx(t2, d3, outV)

    # uncompute t2
    qc.ccx(t1, d2, t2)
    qc.cx(d2, t2)
    qc.cx(t1, t2)
    # uncompute t1
    qc.ccx(d0, d1, t1)
    qc.cx(d1, t1)
    qc.cx(d0, t1)

    # All ancilla t1,t2 are restored to 0.
    return

def reversible_8to3(bits):
    """
    bits: list/tuple of 8 bits [D0..D7] (D7 highest priority)
    Returns: QuantumCircuit with outputs measured into classical register [A,B,C,V]
    Qubit mapping (indexes):
      q0..q7   : D0..D7 (inputs)
      q8       : A1 (lower 4->2 A)
      q9       : B1
      q10      : V1
      q11      : A2 (upper 4->2 A)
      q12      : B2
      q13      : V2
      q14      : ancilla t1 (shared)
      q15      : ancilla t2 (shared)
      q16      : A_out
      q17      : B_out
      q18      : C_out
      q19      : V_out
    """
    if len(bits) != 8:
        raise ValueError("bits must be length 8")

    q = QuantumRegister(20, 'q')
    c = ClassicalRegister(4, 'c')   # we'll map c[0]=A, c[1]=B, c[2]=C, c[3]=V
    qc = QuantumCircuit(q, c)

    # set inputs
    for i in range(8):
        if bits[i] == 1:
            qc.x(q[i])

    # Lower 4->2: D0..D3 -> A1(q8), B1(q9), V1(q10) using t1=q14,t2=q15
    reversible_4to2([q[0], q[1], q[2], q[3]], q[8], q[9], q[10], q[14], q[15], qc)

    # Upper 4->2: D4..D7 -> A2(q11), B2(q12), V2(q13) using same ancillas (they were restored)
    reversible_4to2([q[4], q[5], q[6], q[7]], q[11], q[12], q[13], q[14], q[15], qc)

    # C_out = copy V2 -> q18
    qc.cx(q[13], q[18])

    # B_out = B2 OR (B1 AND NOT V2)  -> q17
    # compute NOT V2 in q14 (t1) without flipping q13:
    qc.x(q[13])
    qc.cx(q[13], q[14])   # q14 = NOT V2
    qc.x(q[13])

    # t2 (q15) = B1 AND q14
    qc.ccx(q[9], q[14], q[15])

    # B_out in q17 = B2 OR t2
    qc.cx(q[12], q[17])
    qc.cx(q[15], q[17])
    qc.ccx(q[12], q[15], q[17])

    # uncompute t2 (q15)
    qc.ccx(q[9], q[14], q[15])

    # uncompute q14 (NOT V2)
    qc.x(q[13])
    qc.cx(q[13], q[14])
    qc.x(q[13])

    # A_out = A2 OR (A1 AND NOT V2)  -> q16
    # compute NOT V2 again into q14
    qc.x(q[13])
    qc.cx(q[13], q[14])
    qc.x(q[13])

    # t2 (q15) = A1 AND q14
    qc.ccx(q[8], q[14], q[15])

    # A_out in q16 = A2 OR t2
    qc.cx(q[11], q[16])
    qc.cx(q[15], q[16])
    qc.ccx(q[11], q[15], q[16])

    # uncompute t2
    qc.ccx(q[8], q[14], q[15])

    # uncompute q14
    qc.x(q[13])
    qc.cx(q[13], q[14])
    qc.x(q[13])

    # V_out = V1 OR V2 -> q19
    qc.cx(q[10], q[19])
    qc.cx(q[13], q[19])
    qc.ccx(q[10], q[13], q[19])

    # Measure final outputs A,B,C,V into classical bits c[0..3]
    qc.measure(q[16], c[0])  # A
    qc.measure(q[17], c[1])  # B
    qc.measure(q[18], c[2])  # C
    qc.measure(q[19], c[3])  # V

    return qc


if __name__ == "__main__":
    # Example: ask user for 8 input bits
    user_input = input("Enter D0 D1 D2 D3 D4 D5 D6 D7 (space separated 0/1): ")
    bits = list(map(int, user_input.strip().split()))
    if len(bits) != 8 or any(b not in [0,1] for b in bits):
        raise SystemExit("Please provide exactly 8 bits 0/1")
    print("\nUser Input:")
    for i in range(8):
        print(f"D{i} = {bits[i]}")


    qc = reversible_8to3(bits)
    print("\nInput Vector:")
    print("D7 D6 D5 D4 D3 D2 D1 D0")
    print(" ".join(str(b) for b in reversed(bits)))

    # show circuit depth & size before transpile
    print("Raw circuit depth:", qc.depth(), "size:", qc.size(), "gate count:", qc.count_ops())

    # run on qasm_simulator with optimizations
    # backend = Aer.get_backend('qasm_simulator')
    # transpiled = transpile(qc, backend, optimization_level=3)
    # print("Transpiled depth:", transpiled.depth(), "size:", transpiled.size(), "ops:", transpiled.count_ops())

    # job = backend.run(transpiled, shots=1024)
    # result = job.result()
    # counts = result.get_counts()
    # print("Counts:", counts)
    # import matplotlib.pyplot as plt

    # print("\n--- Showing Logical Circuit ---")
    # fig1 = qc.draw('mpl', fold=-1)
    # plt.show()

    # print("\n--- Showing Transpiled Circuit ---")
    # fig2 = transpiled.draw('mpl', fold=-1)
    # plt.show()


    # # interpret output bitstring (Qiskit prints classical bits as c3 c2 c1 c0)
    # sample = max(counts, key=counts.get)
    # # sample is 'c3 c2 c1 c0' => V C B A order
    # print("Raw key (c3 c2 c1 c0):", sample)
    # sample = max(counts, key=counts.get)
    # A = sample[1]
    # B = sample[2]
    # C = sample[3]
    # V = sample[0]

    # print("\nOutput (A B C V):")
    # print(A, B, C, V)
    # print("A =", sample[1], " B =", sample[2], " C =", sample[3], " V =", sample[0])
    # #print("\nOutput (A B C V):", sample[3], sample[2], sample[1], sample[0])

    # #transpiled.draw('mpl', fold=-1)
    # #qc.draw('mpl')




User Input:
D0 = 0
D1 = 0
D2 = 0
D3 = 0
D4 = 0
D5 = 0
D6 = 0
D7 = 1

Input Vector:
D7 D6 D5 D4 D3 D2 D1 D0
1 0 0 0 0 0 0 0
Raw circuit depth: 60 size: 85 gate count: OrderedDict({'cx': 39, 'ccx': 25, 'x': 17, 'measure': 4})


In [4]:
#8-3 in real devices 

from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit import transpile

# Initialize service (make sure you're logged in first)
service = QiskitRuntimeService()

# Choose least busy backend with >=9 qubits
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=9)
print(f"Running on: {backend.name}")

# Transpile for hardware
transpiled_qc = transpile(qc, backend, optimization_level=1)

options = {"default_shots": 8000}

sampler = Sampler(mode=backend, options=options)
job = sampler.run([transpiled_qc])

print(f"Job submitted! Job ID: {job.job_id()}")

result = job.result()

# Correct way to get counts
counts = result[0].data.c.get_counts()

print("Final Counts:", counts)


qiskit_runtime_service.__init__:WARNING:2026-02-17 00:06:01,882: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService().
qiskit_runtime_service.backends:WARNING:2026-02-17 00:06:03,516: Loading instance: open-instance, plan: open
qiskit_runtime_service.backends:WARNING:2026-02-17 00:06:08,120: Using instance: open-instance, plan: open


Running on: ibm_torino
Job submitted! Job ID: d69m76ro1r9c73fmgnpg
Final Counts: {'1001': 651, '0111': 366, '1111': 1166, '1101': 515, '1011': 1469, '0011': 428, '0110': 278, '0010': 267, '1010': 755, '0100': 132, '0101': 189, '1100': 275, '1000': 304, '0000': 148, '1110': 828, '0001': 229}
